# Tips for task 4a) in worksheet 03

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 1</h2>
    <details>
    <summary>Click here!</summary>
    
  To initialize your weights you require the weights for convolutional and pooling blocks with a size of $(16,2)$ as well as the weights for the fully connected layer in the end which acts on two qubits and therefore has $4^2-1$ parameters.
  
</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 2</h2>
    <details>
    <summary>Click here!</summary>
    
  Remember to use the PennyLane numpy API for generating your weights and defining `requires_grad=True`.
  
</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 3</h2>
    <details>
    <summary>Click here!</summary>
    
  To compute the costs ensure to have a direct dependency between the qnode output and your cost value. Please also ensure to use the PennyLane numpy again.
  
</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tip 4</h2>
    <details>
    <summary>Click here!</summary>
    
  When performing the `cost_and_step` method from the optimizer, your updated weights are returned in a list. So you need to extract them properly.

  ```python
  updated_weights, current_cost = optimizer.step_and_cost(
        lambda w, wl: compute_cost(w, wl, x_train, y_train), weights, weights_last)
  weights = updated_weights[0]
  weights_last = updated_weights[1]
  ```
  
</details>
</div>

<div class="alert alert-success">
  <h2><i class="fas fa-check" style="font-size:36px"></i> &nbsp; Exemplary Solution </h2>
    <details>
    <summary>Click here!</summary>

  ```python
  def init_weights():
    """Initializes random weights for the QCNN model."""
    weights = pnp.random.normal(loc=0, scale=1, size=(16, 2), requires_grad=True)
    weights_last = pnp.random.normal(loc=0, scale=1, size=4 ** 2 - 1, requires_grad=True)
    return weights, weights_last

  def compute_cost(weights, weights_last, features, labels):
    """Computes the cost over the provided features and labels"""
    out = [classifier[idx] for classifier, idx in zip(qcnn_classifier(weights, weights_last, features), labels)]
    return 1.0 - pnp.sum(out) / len(labels)

  [...]

  # Load the data
  x_train, y_train, x_test, y_test = load_digits_data(n_train, n_test, rng)

  # init weights and optimizer
  weights, weights_last = init_weights()
  optimizer = qml.AdamOptimizer(stepsize=0.1)

  # data containers
  train_cost_epochs, test_cost_epochs = [
    compute_cost(weights, weights_last, x_train, y_train)], [
        compute_cost(weights, weights_last, x_test, y_test)]

  for step in range(n_epochs):
    # Training step with (adam) optimizer
    updated_weights, current_cost = optimizer.step_and_cost(
        lambda w, wl: compute_cost(w, wl, x_train, y_train), weights, weights_last)
    weights = updated_weights[0]
    weights_last = updated_weights[1]

    train_cost_epochs.append(current_cost)
    test_cost_epochs.append(compute_cost(weights, weights_last, x_test, y_test))

    if step % 10 == 0:
        print(f"Epoch {step}: Train Cost = {train_cost_epochs[-1]:.4f}, Test Cost = {test_cost_epochs[-1]:.4f}")
  ```
</details>
</div>